In [6]:
import pandas as pd
erp = pd.read_excel('erp.xlsx')
web = pd.read_excel('web.xlsx')
liaison = pd.read_excel('liaison.xlsx')
print(erp.shape, web.shape, liaison.shape)

(825, 6) (1513, 29) (825, 2)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [7]:
import pandas as pd
import pandera as pa
from pandera import Column, Check, DataFrameSchema
import re
from html import unescape

# Liste des product_id dont le prix inférieur au prix d'achat
# a été explicitement autorisé par Nicolas.
# À renseigner avec les identifiants réellement validés.
DESTOCKAGES_AUTORISES = set()

# Stockage des anomalies détectées.
anomalies = []


def enregistrer_anomalie(
    regle, gravite, table, lignes, message
):
    """Enregistre les lignes en anomalie sans modifier les données."""
    for index in lignes:
        anomalies.append({
            "Règle": regle,
            "Gravité": gravite,
            "Table": table,
            "Index": index,
            "Message": message
        })


def executer_schema(schema, df, nom_table, gravites):
    """
    Exécute un schéma en mode lazy et transforme les erreurs
    Pandera en lignes de rapport.
    """
    try:
        schema.validate(df, lazy=True)
    except pa.errors.SchemaErrors as exc:
        details = exc.failure_cases

        for _, erreur in details.iterrows():
            check = str(erreur.get("check", ""))
            colonne = erreur.get("column", None)
            index = erreur.get("index", None)

            # Les identifiants R1, R2... sont présents dans le nom
            # du contrôle pour permettre de retrouver la règle.
            match = re.search(r"R\d+", check)
            if not match:
                continue

            regle = match.group(0)
            gravite = gravites.get(regle, "Avertissement")

            anomalies.append({
                "Règle": regle,
                "Gravité": gravite,
                "Table": nom_table,
                "Index": index,
                "Colonne": colonne,
                "Message": str(erreur.get("failure_case", check))
            })


def prix_positif(df):
    """R1 : le prix de vente doit être strictement positif."""
    return df["price"].notna() & (df["price"] > 0)


def stock_non_negatif(df):
    """R2 : le stock ne doit jamais être négatif."""
    return (
        df["stock_quantity"].notna()
        & (df["stock_quantity"] >= 0)
    )


def statut_stock_coherent(df):
    """R3 : statut en stock cohérent avec la quantité disponible."""
    statut = (
        df["stock_status"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Adapter ces libellés aux valeurs réelles de l'ERP.
    en_stock = statut.isin(["instock", "en stock"])
    rupture = statut.isin([
        "outofstock", "out of stock", "rupture"
    ])

    return (
        (en_stock & (df["stock_quantity"] > 0))
        | (rupture & (df["stock_quantity"] == 0))
    )


def prix_superieur_achat(df):
    """
    R4 : le prix de vente doit être au moins égal au prix d'achat,
    sauf pour les produits de déstockage autorisés.
    """
    exception = df["product_id"].isin(DESTOCKAGES_AUTORISES)

    return (
        df["price"].notna()
        & df["purchase_price"].notna()
        & (
            (df["price"] >= df["purchase_price"])
            | exception
        )
    )


schema_erp = DataFrameSchema(
    columns={
        "product_id": Column(int, nullable=False, unique=True),
        "onsale_web": Column(
            int, Check.isin([0, 1]), nullable=False
        ),
        "price": Column(float, nullable=False),
        "stock_quantity": Column(int, nullable=False),
        "stock_status": Column(str, nullable=False),
        "purchase_price": Column(float, nullable=False),
    },
    checks=[
        Check(prix_positif, name="R1_prix_positif"),
        Check(stock_non_negatif, name="R2_stock_non_negatif"),
        Check(statut_stock_coherent, name="R3_statut_stock"),
        Check(prix_superieur_achat, name="R4_prix_achat"),
    ],
    strict=False,
    coerce=False
)


def reference_numerique(df):
    """R5 : les SKU renseignés doivent avoir un format numérique."""
    sku = df["sku"].astype("string").str.strip()
    return sku.notna() & sku.str.fullmatch(r"\d+", na=False)


def reference_presente(df):
    """R9 : chaque produit doit avoir une référence."""
    sku = df["sku"].astype("string").str.strip()
    return sku.notna() & sku.ne("")


def ventes_non_negatives(df):
    """R9 : les ventes cumulées ne peuvent pas être négatives."""
    return (
        df["total_sales"].notna()
        & (df["total_sales"] >= 0)
    )


def titres_descriptions_sans_html(df):
    """R11 : les titres et descriptions ne doivent pas contenir de HTML."""
    colonnes = [
        c for c in ["post_title", "post_excerpt"]
        if c in df.columns
    ]

    resultat = pd.Series(True, index=df.index)

    for colonne in colonnes:
        texte = df[colonne].astype("string")
        resultat &= ~texte.str.contains(
            r"<[^>]+>", regex=True, na=False
        )

    return resultat


schema_web = DataFrameSchema(
    columns={
        "sku": Column(object, nullable=False),
        "total_sales": Column(float, nullable=False),
        "post_title": Column(str, nullable=True),
        "post_excerpt": Column(str, nullable=True),
    },
    checks=[
        Check(reference_numerique, name="R5_sku_numerique"),
        Check(reference_presente, name="R9_sku_present"),
        Check(ventes_non_negatives, name="R9_ventes_non_negatives"),
        Check(titres_descriptions_sans_html, name="R11_html_absent"),
    ],
    strict=False,
    coerce=False
)


def id_web_numerique(df):
    """R5 : les identifiants web de la liaison sont numériques."""
    identifiant = df["id_web"].astype("string").str.strip()
    return (
        identifiant.isna()
        | identifiant.eq("")
        | identifiant.str.fullmatch(r"\d+", na=False)
    )


schema_liaison = DataFrameSchema(
    columns={
        "id_web": Column(object, nullable=True),
        "product_id": Column(int, nullable=False),
    },
    checks=[
        Check(id_web_numerique, name="R5_id_web_numerique"),
    ],
    strict=False,
    coerce=False
)


# Gravité issue du catalogue de règles.
GRAVITES = {
    "R1": "Bloquante",
    "R2": "Bloquante",
    "R3": "Avertissement",
    "R4": "Bloquante",
    "R5": "Bloquante",
    "R6": "Avertissement",
    "R7": "Avertissement",
    "R8": "Bloquante",
    "R9": "Bloquante",
    "R10": "Avertissement",
    "R11": "Avertissement",
    "R12": "Avertissement",
    "R13": "Bloquante",
    "R14": "Avertissement",
    "R15": "Avertissement",
}


def ajouter(regle, table, index, message):
    """Ajoute une anomalie au rapport central."""
    anomalies.append({
        "Règle": regle,
        "Gravité": GRAVITES[regle],
        "Table": table,
        "Index": index,
        "Message": message,
    })


def verifier_inter_tables(erp, web, liaison):
    """Exécute les contrôles de cohérence entre les trois tables."""

    # Normalisation temporaire des références pour comparer les tables.
    # Les DataFrames d'origine ne sont pas modifiés.
    e = erp.copy()
    w = web.copy()
    l = liaison.copy()

    e["_product_id"] = e["product_id"].astype("string").str.strip()
    l["_product_id"] = l["product_id"].astype("string").str.strip()
    l["_id_web"] = l["id_web"].astype("string").str.strip()
    w["_sku"] = w["sku"].astype("string").str.strip()

    # Références web réellement renseignées.
    ids_liaison = set(
        l.loc[
            l["_id_web"].notna() & l["_id_web"].ne(""),
            "_id_web"
        ]
    )
    skus_web = set(
        w.loc[
            w["_sku"].notna() & w["_sku"].ne(""),
            "_sku"
        ]
    )

    # R5 : un identifiant renseigné doit être numérique dans les deux tables.
    for table, df, colonne in [
        ("web", w, "_sku"),
        ("liaison", l, "_id_web"),
    ]:
        masque = (
            df[colonne].notna()
            & df[colonne].ne("")
            & ~df[colonne].str.fullmatch(r"\d+", na=False)
        )
        for idx in df.index[masque]:
            ajouter("R5", table, idx, "Identifiant web non numérique")

    # R6 et R7 : contrôle entre le statut de vente ERP et la liaison.
    liaison_par_produit = l.groupby("_product_id")["_id_web"].apply(
        lambda s: s.notna().any() and s.ne("").any()
    )

    for idx, ligne in e.iterrows():
        pid = ligne["_product_id"]
        a_un_id_web = bool(liaison_par_produit.get(pid, False))

        if ligne["onsale_web"] == 1 and not a_un_id_web:
            ajouter(
                "R6", "ERP / liaison", idx,
                "Produit en vente web sans identifiant web"
            )

        if ligne["onsale_web"] == 0 and a_un_id_web:
            ajouter(
                "R7", "ERP / liaison", idx,
                "Produit hors vente web avec identifiant web renseigné"
            )

    # R8 : chaque SKU doit apparaître une seule fois dans l'export.
    sku_valide = w["_sku"].notna() & w["_sku"].ne("")
    doublons = w.loc[sku_valide, "_sku"].duplicated(keep=False)

    for idx in w.loc[sku_valide].index[doublons]:
        ajouter(
            "R8", "web", idx,
            f"SKU dupliqué : {w.at[idx, '_sku']}"
        )

    # R10 : colonnes entièrement vides dans l'export.
    # Une colonne numérique remplie de NaN est également considérée vide.
    for colonne in web.columns:
        serie = web[colonne]
        vide = (
            serie.isna()
            | serie.astype("string").str.strip().eq("").fillna(False)
        )

        if vide.all():
            ajouter(
                "R10", "web", colonne,
                f"Colonne entièrement vide : {colonne}"
            )

    # R11 : détecte du HTML dans les titres et descriptions.
    for colonne in ["post_title", "post_excerpt"]:
        if colonne not in web.columns:
            continue

        masque_html = (
            web[colonne]
            .astype("string")
            .str.contains(r"<[^>]+>", regex=True, na=False)
        )

        for idx in web.index[masque_html]:
            ajouter(
                "R11", "web", idx,
                f"Code HTML détecté dans {colonne}"
            )

    # R12 : chaque identifiant de liaison doit exister dans l'export web.
    for idx, ligne in l.iterrows():
        ref = ligne["_id_web"]

        if pd.notna(ref) and ref != "" and ref not in skus_web:
            ajouter(
                "R12", "liaison", idx,
                f"Référence absente de l'export web : {ref}"
            )

    # R13 : tout produit ayant des ventes doit pouvoir être relié à l'ERP.
    produits_erp = set(e["_product_id"].dropna())

    ventes = pd.to_numeric(web["total_sales"], errors="coerce")
    refs_vendues = set(w.loc[ventes > 0, "_sku"].dropna())

    liaison_vendue = l[l["_id_web"].isin(refs_vendues)]

    refs_reliees = set(liaison_vendue["_id_web"].dropna())
    for idx, ligne in liaison_vendue.iterrows():
        if ligne["_product_id"] not in produits_erp:
            ajouter(
                "R13", "ERP / liaison", idx,
                f"Produit vendu absent de l'ERP : {ligne['_product_id']}"
            )

    # R13 : les SKU vendus doivent aussi avoir une correspondance ERP.
    refs_sans_liaison = refs_vendues - refs_reliees
    for ref in refs_sans_liaison:
        ajouter(
            "R13", "web / liaison", ref,
            f"Produit vendu sans correspondance ERP : {ref}"
        )

    # R14 : le contrôle nécessite de connaître les types autorisés
    # et les produits considérés comme non-vins.
    # Voir les limites expliquées sous le code.


def verifier_regles_erp(erp):
    """Contrôles complémentaires sur les règles ERP."""

    # R1 à R4 sont exécutées par le schéma Pandera.
    # R15 ne peut être vérifiée que si une colonne de libellé existe.
    colonnes_libelle = [
        c for c in ["libelle", "product_name", "nom_produit", "name"]
        if c in erp.columns
    ]

    if not colonnes_libelle:
        print(
            "R15 non exécutable : aucune colonne de libellé "
            "n'est présente dans le schéma ERP fourni."
        )
        return

    colonne = colonnes_libelle[0]
    masque = (
        erp[colonne].isna()
        | erp[colonne].astype("string").str.strip().eq("").fillna(False)
    )

    for idx in erp.index[masque]:
        ajouter("R15", "ERP", idx, "Libellé produit vide ou manquant")

In [8]:
%%time
# 1. Valider chaque table indépendamment.
executer_schema(schema_erp, erp, "ERP", GRAVITES)
executer_schema(schema_web, web, "Web", GRAVITES)
executer_schema(schema_liaison, liaison, "Liaison", GRAVITES)

# 2. Appliquer les contrôles complémentaires.
verifier_inter_tables(erp, web, liaison)
verifier_regles_erp(erp)

# 3. Produire un rapport consolidé.
rapport = pd.DataFrame(anomalies)

if rapport.empty:
    print("Aucune anomalie détectée par les contrôles exécutés.")
else:
    rapport = rapport.drop_duplicates()

    ordre = {"Bloquante": 0, "Avertissement": 1}
    rapport["_ordre"] = rapport["Gravité"].map(ordre)

    rapport = (
        rapport.sort_values(["_ordre", "Règle"])
        .drop(columns="_ordre")
        .reset_index(drop=True)
    )

    print("Nombre d'anomalies :", len(rapport))
    print("\nRépartition par règle :")
    print(rapport.groupby(["Règle", "Gravité"]).size())

    print("\nDétail des anomalies :")
    display(rapport)

    bloquantes = rapport[rapport["Gravité"] == "Bloquante"]
    avertissements = rapport[rapport["Gravité"] == "Avertissement"]

    print("\nAnomalies bloquantes :", len(bloquantes))
    print("Avertissements :", len(avertissements))

R15 non exécutable : aucune colonne de libellé n'est présente dans le schéma ERP fourni.
Nombre d'anomalies : 5916

Répartition par règle :
Règle  Gravité      
R1     Bloquante          18
R10    Avertissement       4
R11    Avertissement    3648
R12    Avertissement      20
R2     Bloquante          12
R3     Avertissement      24
R4     Bloquante          42
R5     Bloquante         401
R6     Avertissement       3
R7     Avertissement      21
R8     Bloquante        1428
R9     Bloquante         295
dtype: int64

Détail des anomalies :


,Règle,Gravité,Table,Index,Colonne,Message
0,R1,Bloquante,ERP,151,product_id,4233
1,R1,Bloquante,ERP,469,product_id,5017
2,R1,Bloquante,ERP,739,product_id,6594
3,R1,Bloquante,ERP,151,onsale_web,0
4,R1,Bloquante,ERP,469,onsale_web,0
...,...,...,...,...,...,...
5911,R7,Avertissement,ERP / liaison,661,NaN,Produit hors vente web avec identifiant web re...
5912,R7,Avertissement,ERP / liaison,662,NaN,Produit hors vente web avec identifiant web re...
5913,R7,Avertissement,ERP / liaison,664,NaN,Produit hors vente web avec identifiant web re...
5914,R7,Avertissement,ERP / liaison,687,NaN,Produit hors vente web avec identifiant web re...



Anomalies bloquantes : 2196
Avertissements : 3720
CPU times: user 189 ms, sys: 11.6 ms, total: 201 ms
Wall time: 204 ms
